# vLLM parallelism and continuous-batching benchmark

This notebook benchmarks the nine rank-32 deployment checkpoints at concurrency 1, 8, 32, 128, and 256. It runs three repetitions in two separate modes: real held-out FEN prompts with natural end-of-turn stopping, and exact 128-input/128-output-token hardware comparisons. It does not rerun quality evaluation. Every scenario saves detailed vLLM request timings, commands, logs, configuration, and checkpoint links to Drive.


In [ ]:
# Cell 1: mount Drive and stage the updated package locally
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_BUNDLE_DIR = '/content/drive/MyDrive/colab_drive_bundle'
NESTED_PACKAGE_DIR = f'{DRIVE_BUNDLE_DIR}/fen_move_colab'
PACKAGE_ON_DRIVE = (
    NESTED_PACKAGE_DIR
    if os.path.isfile(f'{NESTED_PACKAGE_DIR}/__init__.py')
    else DRIVE_BUNDLE_DIR
)
WORKSPACE = '/content/fen_move_parallelism_workspace'

required = ['__init__.py', 'run_parallelism_benchmark.py', 'config_parallelism_benchmark.yaml']
missing = [name for name in required if not os.path.isfile(f'{PACKAGE_ON_DRIVE}/{name}')]
if missing:
    raise FileNotFoundError(
        f'Missing {missing} under {PACKAGE_ON_DRIVE}. Upload the updated bundle directly '
        'under MyDrive, replacing the old code files.'
    )

def pip_install(*packages):
    command = [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *packages]
    print('Running:', ' '.join(command), flush=True)
    subprocess.check_call(command)

shutil.rmtree(WORKSPACE, ignore_errors=True)
os.makedirs(WORKSPACE, exist_ok=True)
shutil.copytree(
    PACKAGE_ON_DRIVE,
    f'{WORKSPACE}/fen_move_colab',
    ignore=shutil.ignore_patterns(
        'raw', 'datasets', 'runs', 'vllm_exports', 'vllm_benchmarks',
        'publication_evaluation_v1', 'parallelism_benchmarks_v1',
        '__pycache__', '.ipynb_checkpoints',
    ),
)
os.chdir(WORKSPACE)
if WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)

try:
    token = userdata.get('HF_TOKEN') or userdata.get('hf_token')
except Exception:
    token = None
if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token

subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap', '--format=csv,noheader'])
print(f'Ready. Package source: {PACKAGE_ON_DRIVE}')


In [ ]:
# Cell 2: edit model selection and benchmark settings
import yaml

CONFIG_PATH = f'{WORKSPACE}/parallelism_benchmark_runtime.yaml'
DEFAULT_CONFIG_PATH = f'{WORKSPACE}/fen_move_colab/config_parallelism_benchmark.yaml'
with open(DEFAULT_CONFIG_PATH, encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

# The completed quality evaluation is the source of the exact held-out split and
# checkpoint hashes. Select the newest completed comparison automatically.
quality_root = Path(DRIVE_BUNDLE_DIR) / 'publication_evaluation_v1'
quality_reports = sorted(
    quality_root.glob('publication_eval_*/comparison.json'),
    key=lambda path: path.stat().st_mtime,
)
if not quality_reports:
    raise FileNotFoundError(f'No completed publication comparison under {quality_root}')
quality_comparison = quality_reports[-1]
heldout_files = sorted((quality_comparison.parent / 'testset').glob('heldout_*.jsonl'))
if len(heldout_files) != 1:
    raise FileNotFoundError(
        f'Expected one heldout_*.jsonl beside {quality_comparison}, found {heldout_files}'
    )
CONFIG['data']['quality_comparison_path'] = str(quality_comparison)
CONFIG['data']['testset_path'] = str(heldout_files[0])

# Default: nine rank-32 deployments (3 model sizes x 3 precisions).
# For a smoke test, uncomment the next line, then restore all nine for publication.
# CONFIG['models'] = CONFIG['models'][:1]

# Editable benchmark controls:
# CONFIG['benchmark']['concurrencies'] = [1, 8, 32, 128, 256]
# CONFIG['benchmark']['repetitions'] = 3
# CONFIG['benchmark']['modes'] = ['natural_fen', 'fixed_tokens']
# CONFIG['run']['resume'] = True
# CONFIG['run']['fail_fast'] = False

with open(CONFIG_PATH, 'w', encoding='utf-8') as handle:
    yaml.safe_dump(CONFIG, handle, sort_keys=False)

scenario_count = (
    len(CONFIG['benchmark']['modes'])
    * len(CONFIG['benchmark']['concurrencies'])
    * CONFIG['benchmark']['repetitions']
)
print(f'Models: {len(CONFIG["models"])}')
print(f'Scenarios per model: {scenario_count}')
print(f'Quality evidence: {quality_comparison}')
print(f'Held-out prompts: {heldout_files[0]}')
print(f'Output root: {CONFIG["run"]["output_root"]}')
print(f'Runtime config: {CONFIG_PATH}')


In [ ]:
# Cell 3: install the proven Gemma 4/vLLM stack plus benchmark extras
SERVING_VLLM_VERSION = '0.27.0'
SERVING_TORCH_VERSION = '2.13.0'
SERVING_CUDA_NVCC_VERSION = '13.2.86'
SERVING_TRANSFORMERS_VERSION = '5.14.1'
SERVING_COMPRESSED_TENSORS_VERSION = '0.17.0'

def probe_serving_stack(show_error=False):
    result = subprocess.run(
        [
            sys.executable, '-c',
            (
                'import torch, transformers, vllm, compressed_tensors; '
                'from vllm import LLM, SamplingParams; '
                f'assert vllm.__version__ == \"{SERVING_VLLM_VERSION}\", vllm.__version__; '
                f'assert torch.__version__.split(\"+\", 1)[0] == \"{SERVING_TORCH_VERSION}\", torch.__version__; '
                f'assert transformers.__version__ == \"{SERVING_TRANSFORMERS_VERSION}\", transformers.__version__; '
                f'assert compressed_tensors.__version__ == \"{SERVING_COMPRESSED_TENSORS_VERSION}\", compressed_tensors.__version__; '
                'print(\"Serving stack:\", torch.__version__, torch.version.cuda, '
                'transformers.__version__, vllm.__version__, compressed_tensors.__version__)'
            ),
        ],
        text=True, capture_output=True,
    )
    if result.stdout:
        print(result.stdout, end='')
    if show_error and result.returncode and result.stderr:
        print(result.stderr, file=sys.stderr, end='')
    return result.returncode == 0

def remove_optional_torchcodec():
    # vLLM imports optional video support while exposing LLM. Some Colab
    # CUDA stacks install a TorchCodec wheel whose libnvrtc dependency cannot
    # load. This benchmark is text-only, so remove that optional backend.
    subprocess.run(
        [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchcodec'],
        check=False,
    )

def configure_cuda13_toolkit():
    # FlashInfer compiles SM120 FP4 kernels at first use. A partial pip upgrade
    # can leave 13.3 nvcc beside 13.2 headers; CCCL rejects that combination.
    import importlib.metadata as metadata
    import site
    from pathlib import Path

    target_version = SERVING_CUDA_NVCC_VERSION
    target_minor = '.'.join(target_version.split('.')[:2])
    required_packages = [
        'nvidia-cuda-cccl', 'nvidia-cuda-crt', 'nvidia-cuda-nvcc',
        'nvidia-cuda-nvrtc', 'nvidia-cuda-runtime', 'nvidia-nvjitlink',
        'nvidia-nvvm',
    ]

    # uv/vLLM can resolve CUDA component wheels independently. Force one full
    # CUDA 13.2 Update 2 toolchain, matching torch's cu132 runtime.
    print(f'Aligning CUDA JIT toolchain to {target_version} for FlashInfer SM120...', flush=True)
    pip_install(
        '--upgrade', '--force-reinstall', '--no-deps',
        *[f'{package}=={target_version}' for package in required_packages],
    )
    versions = {}
    for package in required_packages:
        try:
            versions[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            versions[package] = None
    mismatched = {name: version for name, version in versions.items() if version != target_version}
    if mismatched:
        raise RuntimeError(
            'CUDA JIT component installation is inconsistent. Expected all ' +
            f'components at {target_version}, got {mismatched}. Restart the Colab '
            'runtime, run Cell 1, then rerun this cell.'
        )

    import torch
    if torch.version.cuda != target_minor:
        raise RuntimeError(
            f'vLLM installed torch CUDA {torch.version.cuda}, but this notebook requires '
            f'{target_minor}. Restart and rerun Cells 1-5 so the serving stack is recreated.'
        )

    candidates = []
    for site_root in [*site.getsitepackages(), site.getusersitepackages()]:
        nvidia_root = Path(site_root) / 'nvidia'
        if nvidia_root.is_dir():
            candidates.extend(nvidia_root.glob('**/bin/nvcc'))
    nvcc = None
    version_text = ''
    for candidate in candidates:
        probe = subprocess.run([str(candidate), '--version'], text=True, capture_output=True)
        if probe.returncode == 0 and f'release {target_minor}' in probe.stdout:
            nvcc, version_text = candidate.resolve(), probe.stdout.strip()
            break
    if nvcc is None:
        raise RuntimeError(
            f'CUDA {target_minor} nvcc was not found after installing {target_version}. '
            'Native SM120 NVFP4 cannot run safely in this runtime.'
        )

    cuda_home = nvcc.parent.parent
    library_files = {}
    for site_root in [*site.getsitepackages(), site.getusersitepackages()]:
        nvidia_root = Path(site_root) / 'nvidia'
        if not nvidia_root.is_dir():
            continue
        for stem in ('libcudart.so', 'libnvrtc.so', 'libnvJitLink.so', 'libnvvm.so'):
            matches = sorted(nvidia_root.glob(f'**/{stem}*'))
            if matches and stem not in library_files:
                library_files[stem] = matches[-1].resolve()
    missing_libraries = [stem for stem in ('libcudart.so', 'libnvrtc.so') if stem not in library_files]
    if missing_libraries:
        raise RuntimeError(f'Missing CUDA JIT libraries: {missing_libraries}')

    # nvcc's linker consults LIBRARY_PATH, not just LD_LIBRARY_PATH. Create stable
    # unversioned symlinks so the first JIT compile cannot pick stale CUDA 13.3 libs.
    link_dir = Path('/content/flashinfer_cuda132_libs')
    shutil.rmtree(link_dir, ignore_errors=True)
    link_dir.mkdir(parents=True, exist_ok=True)
    for stem, library in library_files.items():
        (link_dir / stem).symlink_to(library)

    os.environ['CUDA_HOME'] = str(cuda_home)
    os.environ['CUDA_PATH'] = str(cuda_home)
    os.environ['CUDA_VERSION'] = target_minor
    os.environ['FLASHINFER_CUDA_VERSION'] = target_minor
    os.environ['PATH'] = os.pathsep.join([str(nvcc.parent), os.environ.get('PATH', '')])
    os.environ['LD_LIBRARY_PATH'] = os.pathsep.join([str(link_dir), os.environ.get('LD_LIBRARY_PATH', '')])
    os.environ['LIBRARY_PATH'] = os.pathsep.join([str(link_dir), os.environ.get('LIBRARY_PATH', '')])
    return {
        'target_version': target_version,
        'package_versions': versions,
        'cuda_home': str(cuda_home),
        'nvcc': str(nvcc),
        'nvcc_version': version_text,
        'link_dir': str(link_dir),
        'libraries': {name: str(path) for name, path in library_files.items()},
    }

def patch_vllm_gemma4_shared_kv_sft_loader():
    # vLLM 0.27.0 registers k_norm on Gemma 4 shared-KV layers, while
    # Transformers correctly omits those unused tensors from SFT saves.
    # Apply the narrow upstream loader fix in this ephemeral Colab runtime.
    import ast
    import hashlib
    import importlib.util
    from pathlib import Path

    spec = importlib.util.find_spec('vllm')
    if spec is None or not spec.submodule_search_locations:
        raise RuntimeError('Cannot locate the installed vLLM package.')
    source_path = Path(next(iter(spec.submodule_search_locations))) / 'model_executor/models/gemma4.py'
    source = source_path.read_text(encoding='utf-8')
    safe_marker = '# codex-gemma4-shared-kv-sft-loader-v2'
    legacy_marker = '# SFT checkpoints omit k_norm.weight for KV-shared layers because those'
    if safe_marker in source:
        return {
            'status': 'already_patched', 'vllm_version': SERVING_VLLM_VERSION,
            'source_path': str(source_path),
            'sha256': hashlib.sha256(source.encode()).hexdigest(),
        }

    backup_path = Path('/tmp/vllm_gemma4_v0_27_0_original.py')
    if legacy_marker in source:
        legacy_fill = '                    params_dict[k_norm_key].fill_(1.0)\n'
        safe_fill = (
            '                    with torch.no_grad():\n'
            '                        params_dict[k_norm_key].fill_(1.0)\n'
        )
        if source.count(legacy_fill) != 1:
            raise RuntimeError(
                'Found the legacy Gemma 4 patch but not its unique fill_ statement; '
                'restart the runtime before retrying.'
            )
        patched = source.replace(legacy_fill, safe_fill, 1)
        patched = patched.replace(
            '        ' + legacy_marker,
            '        ' + safe_marker + '\n        ' + legacy_marker,
            1,
        )
        compile(patched, str(source_path), 'exec')
        source_path.write_text(patched, encoding='utf-8')
        return {
            'status': 'upgraded_legacy_patch', 'vllm_version': SERVING_VLLM_VERSION,
            'source_path': str(source_path), 'backup_path': str(backup_path),
            'sha256_before': hashlib.sha256(source.encode()).hexdigest(),
            'sha256_after': hashlib.sha256(patched.encode()).hexdigest(),
        }

    tree = ast.parse(source, filename=str(source_path))
    model_classes = [
        node for node in tree.body
        if isinstance(node, ast.ClassDef) and node.name == 'Gemma4Model'
    ]
    if len(model_classes) != 1:
        raise RuntimeError(f'Expected one Gemma4Model class, found {len(model_classes)}.')
    load_methods = [
        node for node in model_classes[0].body
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == 'load_weights'
    ]
    if len(load_methods) != 1:
        raise RuntimeError(
            f'Expected one Gemma4Model.load_weights method, found {len(load_methods)}.'
        )
    return_nodes = [
        node for node in load_methods[0].body
        if isinstance(node, ast.Return)
        and isinstance(node.value, ast.Name)
        and node.value.id == 'loaded_params'
    ]
    if len(return_nodes) != 1:
        raise RuntimeError(
            'Refusing to patch an unexpected Gemma4Model.load_weights control flow. '
            f'Expected one direct return of loaded_params, found {len(return_nodes)}.'
        )
    return_node = return_nodes[0]
    source_lines = source.splitlines(keepends=True)
    return_line = source_lines[return_node.lineno - 1]
    indentation = return_line[:len(return_line) - len(return_line.lstrip())]
    if return_node.col_offset != load_methods[0].col_offset + 4:
        raise RuntimeError('The loaded_params return is not at method scope; refusing to patch.')
    insertion = (
        '        # codex-gemma4-shared-kv-sft-loader-v2\n'
        '        # SFT checkpoints omit k_norm.weight for KV-shared layers because those\n'
        '        # layers reuse an earlier KV cache and never execute k_norm. The tensor\n'
        '        # value is therefore irrelevant; initialize it only for loader accounting.\n'
        '        for mod_name, mod in self.named_modules():\n'
        '            if isinstance(mod, Gemma4Attention) and mod.is_kv_shared_layer:\n'
        '                k_norm_key = f"{mod_name}.k_norm.weight"\n'
        '                if k_norm_key in params_dict and k_norm_key not in loaded_params:\n'
        '                    with torch.no_grad():\n'
        '                        params_dict[k_norm_key].fill_(1.0)\n'
        '                    loaded_params.add(k_norm_key)\n'
    )
    if indentation != '        ':
        insertion = ''.join(
            indentation + line[8:] if line.strip() else line
            for line in insertion.splitlines(keepends=True)
        )
    source_lines.insert(return_node.lineno - 1, insertion)
    patched = ''.join(source_lines)
    compile(patched, str(source_path), 'exec')
    if not backup_path.exists():
        backup_path.write_text(source, encoding='utf-8')
    source_path.write_text(patched, encoding='utf-8')
    return {
        'status': 'patched', 'vllm_version': SERVING_VLLM_VERSION,
        'source_path': str(source_path), 'backup_path': str(backup_path),
        'upstream_pr': 'https://github.com/vllm-project/vllm/pull/43812',
        'sha256_before': hashlib.sha256(source.encode()).hexdigest(),
        'sha256_after': hashlib.sha256(patched.encode()).hexdigest(),
    }

if not probe_serving_stack():
    print('Repairing optional TorchCodec before replacing the serving stack...', flush=True)
    remove_optional_torchcodec()

if not probe_serving_stack():
    print('Switching from the export stack to a driver-matched vLLM serving stack...', flush=True)
    pip_install('--upgrade', 'uv')
    subprocess.run([
        sys.executable, '-m', 'pip', 'uninstall', '-y',
        'vllm', 'llmcompressor', 'compressed-tensors',
        'torch', 'torchvision', 'torchaudio', 'torchao', 'torchcodec',
    ], check=False)
    subprocess.check_call([
        sys.executable, '-m', 'uv', 'pip', 'install', '--system', '--upgrade',
        '--torch-backend=auto', f'vllm[bench]=={SERVING_VLLM_VERSION}',
        f'transformers=={SERVING_TRANSFORMERS_VERSION}',
        f'compressed-tensors=={SERVING_COMPRESSED_TENSORS_VERSION}',
        'python-chess', 'pandas', 'pyyaml', 'nvidia-ml-py', 'aiohttp', 'datasets',
    ])
    remove_optional_torchcodec()
    if not probe_serving_stack(show_error=True):
        raise RuntimeError('The vLLM serving stack still cannot import; see the full stderr above.')

CUDA13_TOOLKIT_INFO = configure_cuda13_toolkit()
print('CUDA 13 SM120 JIT toolchain:', json.dumps(CUDA13_TOOLKIT_INFO, indent=2))
VLLM_GEMMA4_PATCH_INFO = patch_vllm_gemma4_shared_kv_sft_loader()
print('Gemma 4 shared-KV SFT loader compatibility:', json.dumps(VLLM_GEMMA4_PATCH_INFO, indent=2))


# A pre-existing matching serving stack may not include the optional benchmark
# dependencies. Install them without changing the pinned serving versions.
bench_probe = subprocess.run(
    ['vllm', 'bench', 'serve', '--help=all'], text=True, capture_output=True
)
if bench_probe.returncode != 0:
    print('Installing vLLM benchmark extras...', flush=True)
    pip_install('--upgrade', 'uv')
    subprocess.check_call([
        sys.executable, '-m', 'uv', 'pip', 'install', '--system', '--upgrade',
        '--torch-backend=auto', f'vllm[bench]=={SERVING_VLLM_VERSION}',
        f'transformers=={SERVING_TRANSFORMERS_VERSION}',
        f'compressed-tensors=={SERVING_COMPRESSED_TENSORS_VERSION}',
    ])
    bench_probe = subprocess.run(
        ['vllm', 'bench', 'serve', '--help=all'], text=True, capture_output=True
    )
if bench_probe.returncode != 0:
    print(bench_probe.stderr, file=sys.stderr)
    raise RuntimeError('vllm bench serve is unavailable after installing benchmark extras.')
print('vLLM online benchmark client is ready.')


In [ ]:
# Cell 4: run resumable online concurrency benchmarks with live output
command = [
    sys.executable, '-u', '-m', 'fen_move_colab.run_parallelism_benchmark',
    '--config', CONFIG_PATH,
]
print('Running:', ' '.join(command), flush=True)
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(
    command,
    cwd=WORKSPACE,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, command)


In [ ]:
# Cell 5: inspect the latest saved comparison
import pandas as pd
from IPython.display import display

output_root = Path(CONFIG['run']['output_root'])
comparisons = sorted(
    output_root.glob('parallelism_*/comparison.json'),
    key=lambda path: path.stat().st_mtime,
)
if not comparisons:
    raise FileNotFoundError(f'No comparison.json under {output_root}')
comparison_path = comparisons[-1]
comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
csv_path = comparison_path.parent / 'performance_comparison.csv'
if csv_path.is_file():
    frame = pd.read_csv(csv_path)
    preferred = [
        'model', 'mode', 'concurrency', 'repetition', 'completed',
        'request_throughput', 'output_throughput', 'total_token_throughput',
        'p50_ttft_ms', 'p95_ttft_ms', 'p50_e2el_ms', 'p95_e2el_ms',
    ]
    display(frame[[column for column in preferred if column in frame.columns]])
else:
    display([
        {'model': row.get('name'), 'status': row.get('status'), 'error': row.get('error')}
        for row in comparison.get('results', [])
    ])
print(f'Complete benchmark evidence: {comparison_path.parent}')
